In [20]:
pip install scikit-learn

## Tokenization comparison

In [19]:
import re

sample_texts = [
    "AI, search!",
    "I can't log in.",
    "help@example.com",
    "CS4944 version 2.1",
    "",
    "میرا نام احمد ہے۔"
]

def whitespace_tokenize(text):
    return text.split()

def regex_tokenize(text):
    return re.findall(r"\b\w+\b", text, flags=re.UNICODE)

for number, text in enumerate(sample_texts, start=1):
    whitespace_tokens = whitespace_tokenize(text)
    regex_tokens = regex_tokenize(text)

    print(f"EXAMPLE {number}")
    print("Original text:       ", repr(text))
    print("Whitespace tokens:   ", whitespace_tokens)
    print("Whitespace count:    ", len(whitespace_tokens))
    print("Regex tokens:        ", regex_tokens)
    print("Regex count:         ", len(regex_tokens))
    print("-" * 60)

assert regex_tokenize("") == []
print("Empty-string check passed.")

EXAMPLE 1
Original text:        'AI, search!'
Whitespace tokens:    ['AI,', 'search!']
Whitespace count:     2
Regex tokens:         ['AI', 'search']
Regex count:          2
------------------------------------------------------------
EXAMPLE 2
Original text:        "I can't log in."
Whitespace tokens:    ['I', "can't", 'log', 'in.']
Whitespace count:     4
Regex tokens:         ['I', 'can', 't', 'log', 'in']
Regex count:          5
------------------------------------------------------------
EXAMPLE 3
Original text:        'help@example.com'
Whitespace tokens:    ['help@example.com']
Whitespace count:     1
Regex tokens:         ['help', 'example', 'com']
Regex count:          3
------------------------------------------------------------
EXAMPLE 4
Original text:        'CS4944 version 2.1'
Whitespace tokens:    ['CS4944', 'version', '2.1']
Whitespace count:     3
Regex tokens:         ['CS4944', 'version', '2', '1']
Regex count:          4
--------------------------------------------

## Build the FAQ chatbot with TF-IDF


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

faqs = [
    {
        "question": "How do I reset my password?",
        "answer": "Open the sign-in page, select 'Forgot password', and follow the instructions sent to your registered email.",
        "source": "Student IT Help Guide"
    },
    {
        "question": "How can I submit my assignment?",
        "answer": "Open your course page, select the relevant assignment, upload your file, and confirm submission.",
        "source": "Learning Management System Guide"
    },
    {
        "question": "What should I do if I am ill and may submit late?",
        "answer": "Contact your course instructor as soon as possible and explain your situation. Follow the instructor's guidance about late submission.",
        "source": "Course Submission Guidance"
    },

    {
        "question": "How do I connect to campus Wi-Fi?",
        "answer": "Choose the official campus Wi-Fi network and follow the login instructions provided by your institution's IT department.",
        "source": "Campus IT Help Desk"
    },
    {
        "question": "What are the library opening hours?",
        "answer": "Please check the library's official notice or website for the current opening hours.",
        "source": "University Library Information"
    }
]

faq_questions = [item["question"] for item in faqs]
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
faq_matrix = vectorizer.fit_transform(faq_questions)

def answer_question(user_question, threshold=0.15, top_n=2):
    cleaned_question = user_question.strip()

    if not cleaned_question:
        return "Please enter a question."

    for item in faqs:
        if cleaned_question.casefold() == item["question"].casefold():
            return (
                f"Answer: {item['answer']}\\n"
                f"Source: {item['source']}"
            )

    query_vector = vectorizer.transform([cleaned_question])
    scores = cosine_similarity(query_vector, faq_matrix).flatten()
    best_indices = scores.argsort()[::-1][:top_n]

    if len(scores) == 0 or scores[best_indices[0]] < threshold:
        return (
            "I could not find a sufficiently relevant FAQ. "
            "Please rephrase your question or contact the relevant department."
        )

    suggestions = []
    for index in best_indices:
        if scores[index] >= threshold:
            suggestions.append(
                f"- {faqs[index]['question']} "
                f"(similarity: {scores[index]:.3f})"
            )

    return (
        "I found these possible matches. Please confirm which one you mean:\\n"
        + "\\n".join(suggestions)
    )

print("FAQ chatbot is ready.")

FAQ chatbot is ready.


## Execution

In [16]:
while True:
    user_input = input("You: ")

    if user_input.strip().casefold() == "exit":
        print("Chatbot: Goodbye!")
        break

    print("Chatbot:", answer_question(user_input))
    print()

You: How do I reset my password?
Chatbot: Answer: Open the sign-in page, select 'Forgot password', and follow the instructions sent to your registered email.\nSource: Student IT Help Guide

You: How do I connect to campus Wi-Fi?
Chatbot: Answer: Choose the official campus Wi-Fi network and follow the login instructions provided by your institution's IT department.\nSource: Campus IT Help Desk

You: um\
Chatbot: I could not find a sufficiently relevant FAQ. Please rephrase your question or contact the relevant department.

You: exit
Chatbot: Goodbye!
